# 01 · EDA: mengenal data hasil scrape

**Analisis awal, bukan solusi final.** Notebook ini hanya melihat data apa adanya dari satu
run, tanpa mengubah apa pun. Tujuannya menjawab: data apa yang kita punya, seberapa lengkap,
dan masalah apa yang harus dibereskan di notebook 02 sebelum bisa dianalisis.

Urutan notebook: `01_eda` → `02_preprocessing_normalization` → `03_processing` →
`04_insights_visualization`.

In [ ]:
import matplotlib.pyplot as plt
import nbutil as nb
import pandas as pd

nb.use_style()
pd.set_option("display.max_colwidth", 80)

print("Run:", nb.run_id())
products = nb.load_products()
stores = nb.load_stores()
knit = nb.load_knit()
stockists = nb.load_stockists()
print(f"{len(stores):,} toko · {len(products):,} produk · {len(knit):,} produk knitwear")
print(f"{len(stockists):,} stockist Wooden Ships")

## 1. Toko: berapa yang benar-benar terbaca?

Status `ok` berarti katalognya terbaca. Selain itu, toko belum memberi data produk.
Toko yang tidak terbaca **belum diketahui**, bukan tidak relevan.

In [ ]:
status = stores["status"].value_counts().rename_axis("status").reset_index(name="toko")
status["porsi"] = (status["toko"] / len(stores)).map("{:.0%}".format)
status

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 2.4))
order = status.sort_values("toko")
ax.barh(order["status"], order["toko"], height=0.55, color=nb.SERIES[0])
for y, value in enumerate(order["toko"]):
    ax.text(value + 6, y, f"{value}", va="center", color=nb.INK_2, fontsize=8.5)
ax.set_title("Status toko dalam run")
ax.set_xlabel("jumlah toko")
ax.xaxis.grid(True)
ax.set_axisbelow(True)
plt.show()

In [ ]:
readable = stores[stores["status"] == "ok"]
print("Platform toko yang terbaca:")
print(readable["platform"].fillna("tidak diketahui").value_counts().head(8).to_string())
print("\nSegmen dari tahap analisis:")
print(stores["segment"].value_counts().to_string())

## 2. Produk: kolom mana yang terisi?

Kolom `vendor` adalah kunci seluruh analisis ini, karena dari situlah kita tahu brand apa
yang dijual tiap toko.

In [ ]:
fill = pd.DataFrame(
    {
        "kolom": ["vendor", "product_type", "tags", "price_raw", "currency", "title"],
        "terisi": [
            products["vendor"].fillna("").str.strip().ne("").mean(),
            products["product_type"].fillna("").str.strip().ne("").mean(),
            products["tags"].map(lambda v: v is not None and len(v) > 0).mean(),
            products["price_raw"].fillna("").str.strip().ne("").mean(),
            products["currency"].fillna("").str.strip().ne("").mean(),
            products["title"].fillna("").str.strip().ne("").mean(),
        ],
    }
)
fill["terisi"] = fill["terisi"].map("{:.1%}".format)
fill

In [ ]:
print("Sumber data produk:")
print(products["source"].value_counts().to_string())
print("\nMata uang:")
print(products["currency"].fillna("(kosong)").value_counts().head(8).to_string())

## 3. Sebaran jumlah produk per toko

Sebarannya sangat miring: sedikit toko punya ribuan produk. Karena itu, di notebook
berikutnya brand dihitung per **toko**, bukan per produk, supaya satu toko besar tidak
mendominasi.

In [ ]:
per_store = products.groupby("domain").size()
print(per_store.describe().round(0).to_string())

fig, ax = plt.subplots(figsize=(6.4, 2.6))
ax.hist(per_store, bins=40, color=nb.SERIES[0], rwidth=0.9)
ax.set_title("Jumlah produk per toko")
ax.set_xlabel("produk dalam katalog")
ax.set_ylabel("jumlah toko")
ax.yaxis.grid(True)
ax.set_axisbelow(True)
plt.show()

## 4. Knitwear

`is_knitwear` adalah tanda longgar dari bot. `knit_kind` adalah aturan ketat (garment atau
accessory) yang dipakai untuk daftar final. Analisis ini memakai yang ketat.

In [ ]:
loose = int(products["is_knitwear"].sum())
strict = int(products["knit_kind"].fillna("").ne("").sum())
print(f"Tanda longgar (is_knitwear): {loose:,} produk ({loose / len(products):.0%})")
print(f"Aturan ketat (knit_kind):    {strict:,} produk ({strict / len(products):.0%})")
print("\nJenis knitwear:")
print(products.loc[products["knit_kind"].fillna("").ne(""), "knit_kind"].value_counts().to_string())

knit_stores = products[products["knit_kind"].fillna("").ne("")].groupby("domain").size()
print(
    f"\nToko yang punya knitwear: {len(knit_stores)} dari {products['domain'].nunique()} toko terbaca"
)

## 5. Vendor: nama brand ditulis tidak seragam

Toko menulis nama brand dengan cara masing-masing. Ini masalah utama yang harus dibereskan
di notebook 02.

In [ ]:
vendor = products["vendor"].fillna("").str.strip()
named = products[vendor.ne("")].assign(vendor=vendor[vendor.ne("")])
named = named.assign(key=named["vendor"].map(nb.compact))
print(f"Nama vendor mentah yang berbeda:        {named['vendor'].nunique():,}")
print(f"Setelah huruf dan tanda baca disamakan: {named['key'].nunique():,}")

spellings = named.groupby("key")["vendor"].agg(lambda s: sorted(set(s)))
variants = spellings[spellings.map(len) > 1]
print(f"Brand dengan lebih dari satu ejaan:     {len(variants):,}")
variants.sort_values(key=lambda s: s.map(len), ascending=False).head(8)

In [ ]:
top = (
    named.groupby("key")
    .agg(
        brand=("vendor", lambda s: s.value_counts().index[0]),
        toko=("domain", "nunique"),
        produk=("url", "size"),
    )
    .sort_values("toko", ascending=False)
    .head(20)
    .reset_index(drop=True)
)
top

Sebagian "vendor" sebenarnya nama toko itu sendiri (label sendiri), bukan brand yang dibeli
dari pihak lain. Itu harus ditandai, supaya tidak dihitung sebagai brand yang "dijual di
satu toko".

In [ ]:
own = named[named["key"] == named["domain"].map(nb.domain_core)]
print(f"Produk yang vendornya sama dengan nama toko: {len(own):,} ({len(own) / len(named):.0%})")
print(f"Toko yang punya produk seperti itu: {own['domain'].nunique()}")

## 6. Harga

`price_raw` disimpan apa adanya sebagai teks. Tahap analisis sudah membuat kolom `price`
berupa angka untuk produk knitwear. Dua hal yang perlu diperhatikan: mata uangnya campur,
dan ada nilai yang tidak masuk akal.

In [ ]:
print(
    knit.groupby(knit["currency"].fillna("(kosong)"))["price"]
    .describe()[["count", "50%", "max"]]
    .round(0)
    .to_string()
)
usd = knit[(knit["currency"] == "USD") & knit["price"].notna()]
print(f"\nKnitwear dalam USD: {len(usd):,} produk")
print(f"Harga di atas US$2.000: {int((usd['price'] > 2000).sum())} produk")
print(f"Harga nol: {int((usd['price'] <= 0).sum())} produk")

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 2.6))
ax.hist(usd.loc[usd["price"].between(1, 600), "price"], bins=40, color=nb.SERIES[0], rwidth=0.9)
ax.axvline(usd["price"].median(), color=nb.INK_2, linewidth=1)
ax.text(
    usd["price"].median() + 6,
    ax.get_ylim()[1] * 0.9,
    f"median ${usd['price'].median():.0f}",
    color=nb.INK_2,
    fontsize=8.5,
)
ax.set_title("Harga knitwear dalam USD (US$1 sampai US$600)")
ax.set_xlabel("harga (USD)")
ax.set_ylabel("jumlah produk")
ax.yaxis.grid(True)
ax.set_axisbelow(True)
plt.show()

## 7. Kategori dan bahan

Dua dari empat ukuran relevansi (kategori serupa, bahan serupa) bergantung pada kolom ini.

In [ ]:
print(f"Kategori diketahui: {knit['category'].notna().mean():.0%}")
print(f"Bahan diketahui:    {knit['main_material'].notna().mean():.0%}\n")
pd.concat(
    [
        knit["category"].fillna("(tidak diketahui)").value_counts().rename("kategori"),
        knit["main_material"].fillna("(tidak diketahui)").value_counts().rename("bahan utama"),
    ],
    axis=1,
).fillna(0).astype(int)

## 8. Lokasi toko

Tanpa lokasi, sebuah toko tidak bisa masuk hitungan teritori atau celah wilayah.

In [ ]:
located = readable["lat"].notna()
print(
    f"Toko terbaca yang lokasinya diketahui: {int(located.sum())} dari {len(readable)} ({located.mean():.0%})"
)
print(readable["country"].fillna("(tidak diketahui)").value_counts().to_string())

## 9. Data acuan milik kita

Empat berkas di `data/inputs/` adalah pembanding. Yang belum ada akan membatasi analisis.

In [ ]:
reference = {
    "stockists.json (stockist kita)": nb.INPUTS / "stockists.json",
    "accounts.csv (akun Salesforce)": nb.INPUTS / "accounts.csv",
    "brands.csv (brand yang dipantau)": nb.INPUTS / "brands.csv",
    "price_points.csv (harga Wooden Ships)": nb.INPUTS / "price_points.csv",
}
pd.DataFrame(
    {
        "berkas": reference.keys(),
        "ada": ["ya" if p.exists() else "BELUM" for p in reference.values()],
    }
)

In [ ]:
print("Stockist per negara bagian (10 teratas):")
print(stockists["prov_state"].value_counts().head(10).to_string())
print(f"\nStockist dengan koordinat: {int(stockists['lat'].notna().sum())} dari {len(stockists)}")

## Ringkasan EDA

Sel di bawah mencetak temuan dari data yang sedang dimuat, jadi angkanya ikut berubah bila
run-nya diganti.

In [ ]:
ok = int((stores["status"] == "ok").sum())
print(f"1. Hanya {ok} dari {len(stores)} toko ({ok / len(stores):.0%}) yang katalognya terbaca.")
print(
    f"2. Vendor terisi di {products['vendor'].fillna('').str.strip().ne('').mean():.0%} produk, jadi peta brand dan toko bisa dibangun."
)
print(f"3. {len(variants):,} brand ditulis dengan lebih dari satu ejaan: perlu normalisasi nama.")
print(
    f"4. {own['domain'].nunique()} toko menjual label sendiri: perlu ditandai, bukan dihitung sebagai brand."
)
print(
    f"5. Harga knitwear campur mata uang ({knit['currency'].nunique()} mata uang) dan ada nilai ekstrem: bandingkan USD saja, buang nilai tak wajar."
)
print(
    f"6. Bahan diketahui di {knit['main_material'].notna().mean():.0%} knitwear, kategori di {knit['category'].notna().mean():.0%}."
)
print(f"7. Lokasi diketahui untuk {located.mean():.0%} toko terbaca.")
missing = [name.split(" ")[0] for name, path in reference.items() if not path.exists()]
print(f"8. Data acuan yang belum ada: {', '.join(missing) if missing else 'tidak ada'}.")